# House Price Prediction Project

This is a starter notebook scaffold. Due to response size limits, a complete 500+ line notebook cannot be generated in one chat response.

In [ ]:
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


In [ ]:

df=pd.read_csv('houses_improved.csv')
df.head()

In [ ]:
categorical_cols = [
    "Construction_Materials",
    "Housing_Typology",
    "Land_Value_Grading",
    "Type_of_Nearest_Road",
]

# Dropdown choices for the UIs, taken straight from the data
category_choices = {col: sorted(df[col].unique().tolist()) for col in categorical_cols}

In [ ]:
# One-Hot Encoding (no drop_first, matches the notebook)
# ---------------------------------------------------------------------------
df_ohe = pd.get_dummies(df, columns=categorical_cols, dtype=int)
X_ohe = df_ohe.drop(columns=["Price_ETB"])
y = df_ohe["Price_ETB"]

# Label Encoding (FIX: capture each encoder into label_encoders)
# ---------------------------------------------------------------------------
df_le = df.copy()
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df_le[col] = le.fit_transform(df_le[col])
    label_encoders[col] = le
X_le = df_le.drop(columns=["Price_ETB"])


In [ ]:

# Train/test splits
# ---------------------------------------------------------------------------
X_tr_ohe, X_te_ohe, y_train, y_test = train_test_split(X_ohe, y, test_size=0.2, random_state=42)
X_tr_le, X_te_le, _, _ = train_test_split(X_le, y, test_size=0.2, random_state=42)

encodings = {
    "One-Hot Encoding": (X_tr_ohe, X_te_ohe, X_ohe.columns),
    "Label Encoding": (X_tr_le, X_te_le, X_le.columns),
}

scalers = {
    "StandardScaler": StandardScaler(),
    "MinMaxScaler": MinMaxScaler(),
    "None (Unscaled)": None,
}


def get_models():
    return {
        "Linear Regression": LinearRegression(),
        "Ridge Regression": Ridge(alpha=1.0),
        "Lasso Regression": Lasso(alpha=100.0),
        "Support Vector Regression (SVR)": SVR(kernel="rbf", C=1000),
        "Random Forest Regressor (RFR)": RandomForestRegressor(n_estimators=100, random_state=42),
        "Gradient Boosting (GBM)": GradientBoostingRegressor(n_estimators=100, random_state=42),
        "Perceptron Regressor (MLP)": MLPRegressor(hidden_layer_sizes=(100,), max_iter=1000, random_state=42),
    }



In [ ]:
# Train every combination
# ---------------------------------------------------------------------------
all_results = []
trained_pipelines = {}

for enc_name, (X_tr, X_te, feature_cols) in encodings.items():
    for scale_name, scaler_obj in scalers.items():
        if scaler_obj is not None:
            scaler = scaler_obj.__class__()
            X_tr_proc = scaler.fit_transform(X_tr)
            X_te_proc = scaler.transform(X_te)
        else:
            scaler = None
            X_tr_proc = X_tr.values
            X_te_proc = X_te.values

        for model_name, model in get_models().items():
            try:
                model.fit(X_tr_proc, y_train)
                predictions = model.predict(X_te_proc)

                mae = mean_absolute_error(y_test, predictions)
                rmse = np.sqrt(mean_squared_error(y_test, predictions))
                r2 = r2_score(y_test, predictions)

                key = f"{model_name} | {enc_name} | {scale_name}"

                trained_pipelines[key] = {
                    "model": model,
                    "scaler": scaler,
                    "encoding": enc_name,
                    "columns": feature_cols,
                    "algorithm": model_name,
                    "scaler_name": scale_name,
                }

                all_results.append({
                    "Pipeline Key": key,
                    "Algorithm": model_name,
                    "Encoding": enc_name,
                    "Scaler": scale_name,
                    "R² Score": round(r2, 4),
                    "RMSE (ETB)": round(rmse, 2),
                    "MAE (ETB)": round(mae, 2),
                })

                print(f"OK  {key}  (R²={r2:.4f})")

            except Exception as e:
                print(f"FAIL  {model_name} | {enc_name} | {scale_name}  -> {e}")

results_df = (
    pd.DataFrame(all_results)
    .sort_values(by="R² Score", ascending=False)
    .reset_index(drop=True)
)

best_pipeline_key = results_df.iloc[0]["Pipeline Key"]

best_pipeline = trained_pipelines[best_pipeline_key]

print("\n")
print("=" * 60)
print(" BEST PIPELINE")
print("=" * 60)
print(best_pipeline_key)
print(f"R² Score : {results_df.iloc[0]['R² Score']}")



Continue by inserting the preprocessing, model training, evaluation, and Gradio UI sections discussed.

In [ ]:
# ---------------------------------------------------------------------------
# Save everything the apps need
# ---------------------------------------------------------------------------
bundle = {
    "trained_pipelines": trained_pipelines,
    "results_df": results_df,
    "categorical_cols": categorical_cols,
    "label_encoders": label_encoders,
    "best_pipeline_key": best_pipeline_key,
    "category_choices": category_choices,
}

joblib.dump(bundle, "pipeline_bundle.pkl")
print("\nSaved pipeline_bundle.pkl")
print(f"Best pipeline: {best_pipeline_key}")
print(f"Best R²: {results_df.iloc[0]['R² Score']}")

In [ ]:


import joblib
import pandas as pd
import gradio as gr
import plotly.express as px

bundle = joblib.load("pipeline_bundle.pkl")
trained_pipelines = bundle["trained_pipelines"]
results_df = bundle["results_df"]
categorical_cols = bundle["categorical_cols"]
label_encoders = bundle["label_encoders"]
best_pipeline_key = bundle["best_pipeline_key"]
category_choices = bundle["category_choices"]


def predict_price(
    pipeline_key,
    rooms,
    site_area,
    built_area,
    years,
    cbd,
    bus,
    schools,
    material,
    typology,
    grading,
    road,
):
    pipe = trained_pipelines[pipeline_key]
    model = pipe["model"]
    scaler = pipe["scaler"]
    enc_type = pipe["encoding"]

    numeric_dict = {
        "Number_of_Rooms": rooms,
        "Site_Area_sqm": site_area,
        "Built_Area_sqm": built_area,
        "Property_Years": years,
        "Proximity_to_CBD_km": cbd,
        "Proximity_to_Bus_Station_km": bus,
        "Proximity_to_Schools_km": schools,
    }
    categorical_dict = {
        "Construction_Materials": material,
        "Housing_Typology": typology,
        "Land_Value_Grading": grading,
        "Type_of_Nearest_Road": road,
    }

    if enc_type == "One-Hot Encoding":
        # Build the one-hot row directly against the trained columns rather
        # than calling pd.get_dummies on a single row (which breaks when
        # combined with drop_first / sparse categories).
        single_encoded = pd.DataFrame(0, index=[0], columns=pipe["columns"])
        for col, val in numeric_dict.items():
            if col in single_encoded.columns:
                single_encoded.at[0, col] = val
        for col, val in categorical_dict.items():
            dummy_col = f"{col}_{val}"
            if dummy_col in single_encoded.columns:
                single_encoded.at[0, dummy_col] = 1
    else:
        row = {**numeric_dict, **categorical_dict}
        single_encoded = pd.DataFrame([row])[pipe["columns"]]
        for col in categorical_cols:
            le = label_encoders[col]
            val = single_encoded[col].iloc[0]
            single_encoded[col] = le.transform([val])[0] if val in le.classes_ else 0

    X_input = scaler.transform(single_encoded) if scaler is not None else single_encoded.values
    pred = model.predict(X_input)[0]
    return f"{pred:,.2f} ETB"


with gr.Blocks(theme=gr.themes.Soft(primary_hue="blue")) as app:
    gr.Markdown("#  House Value / Price Predictor")

    with gr.Tab("Predict"):
        pipeline_dropdown = gr.Dropdown(
            choices=list(trained_pipelines.keys()),
            value=best_pipeline_key,
            label="Model",
        )

        with gr.Row():
            with gr.Column():
                rooms = gr.Number(value=4, label="Rooms")
                site_area = gr.Number(value=250, label="Site Area (sqm)")
                built_area = gr.Number(value=150, label="Built Area (sqm)")
                years = gr.Number(value=8, label="Property Age (yrs)")

            with gr.Column():
                cbd = gr.Number(value=2.5, label="Distance to CBD (km)")
                bus = gr.Number(value=0.5, label="Distance to Bus Station (km)")
                schools = gr.Number(value=1.2, label="Distance to Schools (km)")
                road = gr.Dropdown(category_choices["Type_of_Nearest_Road"], label="Nearest Road Type")

            with gr.Column():
                material = gr.Dropdown(category_choices["Construction_Materials"], label="Construction Material")
                typology = gr.Dropdown(category_choices["Housing_Typology"], label="Housing Typology")
                grading = gr.Dropdown(category_choices["Land_Value_Grading"], label="Land Value Grading")

        predict_btn = gr.Button("Estimate Price", variant="primary")
        output_price = gr.Textbox(label="Estimated Price", interactive=False)

        predict_btn.click(
            fn=predict_price,
            inputs=[pipeline_dropdown, rooms, site_area, built_area, years,
                    cbd, bus, schools, material, typology, grading, road],
            outputs=output_price,
        )

    with gr.Tab("Benchmarks"):
        with gr.Row():
            gr.Textbox(value=results_df.iloc[0]["Algorithm"], label="Best Algorithm", interactive=False)
            gr.Textbox(value=f"{results_df.iloc[0]['R² Score']}", label="Best R² Score", interactive=False)
            gr.Textbox(value=f"{results_df.iloc[0]['RMSE (ETB)']:,.2f} ETB", label="Lowest RMSE", interactive=False)

        fig = px.bar(
            results_df.head(15),
            x="R² Score",
            y="Pipeline Key",
            color="Algorithm",
            orientation="h",
            title="Top 15 Pipeline Configurations (by R² Score)",
        )
        fig.update_layout(yaxis={"categoryorder": "total ascending"})
        gr.Plot(fig)

        with gr.Accordion("Full performance table", open=False):
            gr.Dataframe(results_df, interactive=False)

if __name__ == "__main__":
    app.launch(inline=True)